# HomeTown AR: 3D models from ALL of a product's photos (Hunyuan3D-2 multi-view), for the whole catalogue
Each product has about 4 photos. This notebook looks at every photo, works out which ones are clean studio shots and from which side
(front / side / back), and builds ONE model from all of them (Hunyuan3D-2mv). If it finds only one usable view it falls back to a single-photo model.

**It is slow (about 1 to 2 minutes per product, so about 10 to 15 hours for ~500 products) and a free Colab session stops after a few hours.**
It is built for that: results are saved to your Google Drive as it goes and a new session continues where the last one stopped.

1. Runtime > Change runtime type > **T4 GPU**.
2. Run cell 1 (install), then **Runtime > Restart session**.
3. Run cells 2, 3, 4 (mount Drive, upload `products.json`, run). When the session stops, come back, run 2, 3, 4 again: finished products are skipped.
4. Download the `hometown_models` folder from Drive (or zip it there), then on your computer:
   `node scripts/colab-batch.mjs import models.zip --replace` then `node scripts/bundle-models.mjs`.

`views.json` in the same folder records which photos were used for each product, so you can see why a model came out wrong.

In [ ]:
!nvidia-smi -L
!rm -rf /content/Hunyuan3D-2
!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2 /content/Hunyuan3D-2
%cd /content/Hunyuan3D-2
!pip -q install -r requirements.txt
!pip -q install -e .
!pip -q install rembg onnxruntime
# when this finishes: Runtime > Restart session, then run cell 2 (not this one again)

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, json
OUT = '/content/drive/MyDrive/hometown_models'
os.makedirs(OUT, exist_ok=True)
print('saving to', OUT, '| already done:', len([f for f in os.listdir(OUT) if f.endswith('.glb')]))

In [ ]:
up = files.upload()          # choose products.json (made by: node scripts/colab-batch.mjs export --all --redo)
products = json.load(open('products.json'))
print(len(products), 'products')

In [ ]:
import io, sys, time, requests, torch, trimesh, numpy as np
from PIL import Image
sys.path.insert(0, '/content/Hunyuan3D-2')
from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from transformers import pipeline as hf_pipeline

assert torch.cuda.is_available(), 'Turn on a GPU: Runtime > Change runtime type > T4 GPU'
single = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2mini', subfolder='hunyuan3d-dit-v2-mini', variant='fp16')
multi = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2mv', subfolder='hunyuan3d-dit-v2-mv', variant='fp16')
rembg_ = BackgroundRemover()
clip = hf_pipeline('zero-shot-image-classification', model='openai/clip-vit-base-patch32', device=0)

LABELS = {
    'front': 'a product photo of furniture seen straight from the front on a plain background',
    'side': 'a product photo of furniture seen straight from the side on a plain background',
    'back': 'a product photo of furniture seen from the back on a plain background',
    'angled': 'a product photo of furniture seen from a three-quarter angle on a plain background',
    'room': 'furniture in a decorated room interior, a lifestyle photo',
    'detail': 'a close-up detail of furniture material or hardware',
}
views_path = f'{OUT}/views.json'
views = json.load(open(views_path)) if os.path.exists(views_path) else {}

def studio_score(rgba):
    # a clean cut-out: the object is centred, fills a fair part of the frame and does not touch the edges
    a = np.array(rgba.split()[-1]) > 20
    if a.mean() < 0.12 or a.mean() > 0.92: return 0.0
    edge = np.concatenate([a[0], a[-1], a[:, 0], a[:, -1]]).mean()
    return float(1.0 - edge)

def classify(img):
    r = clip(img, candidate_labels=list(LABELS.values()))
    inv = {v: k for k, v in LABELS.items()}
    return {inv[x['label']]: x['score'] for x in r}

def choose(photos):
    items = []
    for u in photos:
        try:
            im = Image.open(io.BytesIO(requests.get(u, timeout=60).content)).convert('RGB')
        except Exception:
            continue
        cut = rembg_(im)
        sc = classify(im)
        items.append({'url': u, 'cut': cut, 'cls': sc, 'studio': studio_score(cut)})
    good = [i for i in items if i['studio'] > 0.5 and max(i['cls'], key=i['cls'].get) not in ('room', 'detail')]
    def best(k, pool):
        pool = [i for i in pool if max(i['cls'], key=i['cls'].get) == k]
        return max(pool, key=lambda i: i['cls'][k] * i['studio']) if pool else None
    front = best('front', good) or best('angled', good)
    side = best('side', good)
    back = best('back', good)
    return items, good, front, side, back

def tint(mesh, hexcol):
    c = tuple(int(hexcol.lstrip('#')[i:i+2], 16) for i in (0, 2, 4)) + (255,)
    mesh.visual = trimesh.visual.ColorVisuals(mesh=mesh, vertex_colors=np.tile(np.array(c, dtype=np.uint8), (len(mesh.vertices), 1)))
    return mesh

t0 = time.time()
for n, p in enumerate(products):
    out = f"{OUT}/{p['id']}.glb"
    if os.path.exists(out): continue
    try:
        items, good, front, side, back = choose(p.get('photos') or [p['photo']])
        if front is None and items:
            front = max(items, key=lambda i: i['studio'])           # nothing looked like a studio shot: use the cleanest one
        gen = torch.manual_seed(12355)
        used = {}
        if front is not None and (side is not None or back is not None):
            views_in = {'front': front['cut']}
            used['front'] = front['url']
            if side is not None: views_in['left'] = side['cut']; used['left'] = side['url']
            if back is not None: views_in['back'] = back['cut']; used['back'] = back['url']
            mesh = multi(image=views_in, num_inference_steps=50, octree_resolution=380, num_chunks=20000, generator=gen)[0]
            mode = 'multi-view'
        else:
            used['front'] = front['url']
            mesh = single(image=front['cut'], num_inference_steps=50, octree_resolution=380, num_chunks=20000, generator=gen)[0]
            mode = 'single-view'
        mesh = tint(mesh, p.get('dominantHex', '#b8a58f'))
        mesh.export(out)
        views[p['id']] = {'mode': mode, 'used': used, 'photos': len(items)}
        json.dump(views, open(views_path, 'w'), indent=1)
        print(f"{n + 1}/{len(products)} {p['id']}  {mode}  ({len(used)} view(s))  {int(time.time() - t0)}s")
    except Exception as e:
        print('FAILED', p['id'], str(e)[:160])
    torch.cuda.empty_cache()
print('done or stopped: run this cell again to continue')

### When everything is finished
In Drive, right-click the `hometown_models` folder > Download (Drive zips it for you). Rename the download to `models.zip` if needed, then on your computer:
```
node scripts/colab-batch.mjs import models.zip --replace
node scripts/bundle-models.mjs
```
Only about 60 models go inside the app (each adds about 2 MB). The rest are written to `models/published/` for you to host (see docs/DEPLOY.md).